# CD CosMx Resection: Per-FOV LIANA Analysis

Runs LIANA on resection sample CD_R1_B7 (Cos4).
Block-diagonal kNN graph, 100 permutations → per-FOV h5ad outputs.
Results loaded in 08_cd_resection_liana_lr_modules.ipynb for downstream module analysis.

In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/cd_resection"
SAMPLE     = "Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738"
OUTPUT_DIR = DATA_DIR + "/" + SAMPLE + "/liana"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy.external as sce
import liana as li
from scipy.spatial import distance_matrix
import pyreadr
import gc

In [ ]:
samples = [
'Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_760',                
'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',          
'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',       
'Cos1_6KDC_strictCD_R1_B4_0715_11_08_2025_14_20_23_133',     
'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',  
'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',  
'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',                  
'Cos3_6KDC_strictCD_R1_B5_0716_11_08_2025_14_17_56_277',                
'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
'Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738',
'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',         
'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952'  
]

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# Step 1: Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Step 2: Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
prefix = '/path/to/cosmx_data/cd_resection/'

In [ ]:
j = 0
for i in samples:
    print(i)
    '''
    good_slides = pd.read_csv(prefix+i+'/csv/post_good_2slides_obs.csv')
    index_mapper = pd.read_csv(prefix+i+'/csv/post_trt_index_mapper.csv')
    good_slides_index_double = good_slides['Unnamed: 0'].tolist()
    good_slides_index_num = index_mapper[index_mapper['index'].isin(good_slides_index_double)]['Unnamed: 0'].tolist()
    '''
    df = pd.read_csv(prefix+i+'/aucell_anno.csv')
    #df = df[df.index.isin(good_slides_index_num)]
    df['label_clean'] = df['label'].str.rstrip()
    df=df.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')
    #df=df.drop(['Unnamed: 0','cells','label'], axis = 1)
    #df['index'] = good_slides_index_double
    if 'SB' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[2]
        comb = patient+'_'+section
    elif '11808' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        machine = i.split('_')[0]
        comb = patient+'_'+section+'_'+machine
    else:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        comb = patient+'_'+section
    
    df['patient']=patient
    df['patient']=section
    df['patient_sec'] = comb
    print(comb)
    if j == 0:
        merged = df
    else:
        merged = pd.concat([merged,df],ignore_index=True)
    j+=1

In [ ]:
coarse_counts  = merged.groupby(['patient_sec', 'cell category']).size().reset_index(name='count')

# Step 2: Compute total cells per sample
total_per_sample = coarse_counts.groupby('patient_sec')['count'].transform('sum')

# Step 3: Compute proportion
coarse_counts['proportion'] = coarse_counts['count'] / total_per_sample


In [ ]:

# Step 1: Get the sample order sorted by response
sample_order = (
    coarse_counts[['patient_sec']]
    .drop_duplicates()
    .sort_values(by='patient_sec')['patient_sec']
)

# Step 2: Pivot and reorder by sample_order
plot_df = coarse_counts.pivot_table(
    index='patient_sec',
    columns='cell category',
    values='proportion',
    fill_value=0
).loc[sample_order]

# Step 3: Plot
plt.figure(figsize=(12, 7))
plot_df.plot(
    kind='bar',
    stacked=True,
    width=0.6,
    figsize=(12, 7),
    edgecolor='none',
    linewidth=0
)

# Step 4: Styling
plt.xlabel("Sample", fontsize=14)
plt.ylabel("Proportion", fontsize=14)
plt.title("Cell Type Distribution of CD Surgical Resection 6K Samples", fontsize=16, fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=11)
plt.yticks(fontsize=11)
plt.legend(title='Cell Type', title_fontsize=14, fontsize=13, loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(False)

sns.set_style("white")
plt.gca().set_facecolor('white')
plt.gcf().set_facecolor('white')
plt.tight_layout()
plt.show()


In [ ]:
# Now read it with pyreadr
lr = pyreadr.read_r("/path/to/nichenet/lr_network_human_21122021.rds")

In [ ]:
lr_df = next(iter(lr.values()))
lr_df

In [ ]:
lr_df=lr_df[['from','to']]

In [ ]:
lr_df.columns = ['ligand','receptor']

In [ ]:
# loop through samples from 0 to 11
sample = samples[9]
print(sample)

In [ ]:
patient_sec = 'CD_R1_B7'

In [ ]:
adata = sc.read_h5ad(prefix+sample+'/Processed/'+'norm_log1p.h5ad')

In [ ]:
merged_sub = merged[merged['patient_sec']==patient_sec]

In [ ]:
adata.obs['cell_type'] = merged_sub['cell type'].tolist()
adata.obs['cell_type_short'] = merged_sub['cell type short'].tolist()
adata.obs['cell_category'] = merged_sub['cell category'].tolist()
adata.obs['cell_type_general'] = merged_sub['cell type general'].tolist()

In [ ]:
original_cats = list(adata.obs['cell_category'].astype('category').cat.categories)
reordered_cats = [cat for cat in original_cats if cat != 'Cycling'] + ['Cycling']

adata.obs['cell_category'] = pd.Categorical(
    adata.obs['cell_category'],
    categories=reordered_cats,
    ordered=True
) 

In [ ]:
# Create a color palette
coordinates = adata.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("patial Plot for Cell Category")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(prefix+sample+'/Processed/anno.pdf', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Copy spatial_fov to spatial, but flip the Y-axis
coords = adata.obsm['spatial_fov'].copy()

# Flip Y coordinates
y_max = coords[:, 1].max()
y_min = coords[:, 1].min()
coords[:, 1] = y_max + y_min - coords[:, 1]  # Mirror around center

# Set as spatial
adata.obsm['spatial'] = coords

In [ ]:
fovs = adata.obs['fov'].unique().tolist()

In [ ]:
adata_sub = adata[adata.obs['fov']==fovs[0]]

In [ ]:
plot, _ = li.ut.query_bandwidth(coordinates=adata_sub.obsm['spatial'], start=0, end=500, interval_n=25)
plot

In [ ]:
li.ut.spatial_neighbors(adata_sub, bandwidth=200, cutoff=0.1, kernel='gaussian', set_diag=True, max_neighbours=100)

In [ ]:

# Find all Treg indices
treg_mask = adata_sub.obs['cell_type_general'] == 'Treg'
treg_indices = np.where(treg_mask)[0]

print(f"Total Tregs: {len(treg_indices)}")
print(f"Treg indices: {treg_indices.tolist()}")


In [ ]:
li.pl.connectivity(adata_sub, idx=treg_indices[0], size=0.01
                  )

In [ ]:

for i in fovs:
    print(f"Processing FOV: {i}")
    adata_sub = adata[adata.obs['fov'] == i]
    n_cells = adata_sub.n_obs
    
    # Skip FOVs with too few cells
    if n_cells <= 100:  # max_neighbours is 100, so need more than that
        print(f"Skipping FOV {i} (only {n_cells} cells)")
        continue

    # Compute spatial neighbors
    li.ut.spatial_neighbors(
        adata_sub,
        bandwidth=200,
        cutoff=0.1,
        kernel='gaussian',
        set_diag=True,
        max_neighbours=100
    )

    try:
        lrdata_nn = li.mt.bivariate(
            adata_sub,
            resource=lr_df,
            local_name='cosine',
            global_name="morans",
            n_perms=100,
            mask_negatives=False,
            add_categories=True,
            nz_prop=0.05,
            use_raw=False,
            verbose=True
        )

    except ValueError as e:
        if "No features with non-zero proportions" in str(e):
            print(f"⚠️ Skipping FOV {i}: no features with non-zero proportions.")
            gc.collect()
            continue  # 👉 skip this FOV
        else:
            raise  # re-raise unexpected errors

    # Build output path and create directories if needed
    out_path = (
        prefix
        + sample
        + '/Processed/liana/fov_' + i + '.h5ad'
    )
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    del lrdata_nn.uns
    del lrdata_nn.obsm

    # Subset obs safely
    cols_keep = [
        'fov', 'cell_id', 'sample', 'tissue_section',
        'cell_type', 'cell_type_short', 'cell_category', 'cell_type_general'
    ]
    cols_keep = [c for c in cols_keep if c in lrdata_nn.obs.columns]
    lrdata_nn.obs = lrdata_nn.obs[cols_keep]

    # Save
    lrdata_nn.write_h5ad(out_path)
    print(f"Saved FOV {i} -> {out_path}")

    # Clean up memory
    del lrdata_nn
    gc.collect()